In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import argparse

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

In [2]:
clickstream_csv = pd.read_csv("../data/feature_clickstream.csv", sep=";")
clickstream_csv

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
0,63,118,80,121,55,193,111,112,-101,83,...,-16,-81,-126,114,35,85,-73,76,CUS_0x1037,01/01/2023
1,-108,182,123,4,-56,27,25,-6,284,222,...,-14,-96,200,35,130,94,111,75,CUS_0x1069,01/01/2023
2,-13,8,87,166,214,-98,215,152,129,139,...,26,86,171,125,-130,354,17,302,CUS_0x114a,01/01/2023
3,-85,45,200,89,128,54,76,51,61,139,...,172,96,174,163,37,207,180,118,CUS_0x1184,01/01/2023
4,55,120,226,-86,253,97,107,68,103,126,...,76,43,183,159,-26,104,118,184,CUS_0x1297,01/01/2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215371,414,22,72,57,142,192,11,139,24,63,...,179,91,20,189,-35,-19,15,66,CUS_0xdf6,01/12/2024
215372,116,-124,-108,212,-21,227,146,112,186,-65,...,38,226,319,98,9,152,17,14,CUS_0xe23,01/12/2024
215373,237,-3,-49,375,144,41,-170,324,19,266,...,7,102,64,191,124,220,231,75,CUS_0xe4e,01/12/2024
215374,5,67,211,83,207,-41,325,14,-18,41,...,109,266,28,157,131,116,101,131,CUS_0xedd,01/12/2024


In [3]:
attributes_csv = pd.read_csv("../data/features_attributes.csv", sep=";")
attributes_csv

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
0,CUS_0x1000,Alistair Barrf,18,913-74-1218,Lawyer,01/05/2023
1,CUS_0x1009,Arunah,26,063-67-6938,Mechanic,01/01/2025
2,CUS_0x100b,Shirboni,19,#F%$D@*&8,Media_Manager,01/03/2024
3,CUS_0x1011,Schneyerh,44,793-05-8223,Doctor,01/11/2023
4,CUS_0x1013,Cameront,44,930-49-9615,Mechanic,01/12/2023
...,...,...,...,...,...,...
12495,CUS_0xff3,Somervilled,55,#F%$D@*&8,Scientist,01/06/2024
12496,CUS_0xff4,Poornimaf,37,655-05-7666,Entrepreneur,01/12/2024
12497,CUS_0xff6,Shieldsb,19,541-92-8371,Doctor,01/10/2024
12498,CUS_0xffc,Brads,18,226-86-7294,Musician,01/01/2024


In [4]:
financials_csv = pd.read_csv("../data/features_financials.csv", sep=";")
financials_csv

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
0,CUS_0x1000,30625.94,"2,70616E+16",6,5,27,2,"Credit-Builder Loan, and Home Equity Loan",57,26,...,Bad,1562.91,"3,00772E+15",10 Years and 9 Months,Yes,"4,29411E+16",77.31427572208112,High_spent_Medium_value_payments,"4,00361E+16",01/05/2023
1,CUS_0x1009,52312.68_,425039,6,5,17,4,"Not Specified, Home Equity Loan, Credit-Builde...",5,18,...,_,202.68,"4,0287E+13",31 Years and 0 Months,Yes,"1,08366E+16",58.66019164829086,High_spent_Medium_value_payments,"5,08012E+16",01/01/2025
2,CUS_0x100b,113781.38999999998,95497825,1,4,1,0,NaN,14,8,...,Good,1030.2,"2,85929E+15",15 Years and 10 Months,No,0,617.0792665202719,High_spent_Small_value_payments,"5,97899E+15",01/03/2024
3,CUS_0x1011,58918.47,52088725,3,3,17,3,"Student Loan, Credit-Builder Loan, and Debt Co...",27,13,...,Standard,473.14,"2,783E+15",15 Years and 10 Months,Yes,"1,23435E+16",383.35084463651407,Low_spent_Medium_value_payments,"2,94101E+15",01/11/2023
4,CUS_0x1013,98620.98,"7,96242E+15",3,3,6,3,"Student Loan, Debt Consolidation Loan, and Per...",12,9,...,Good,1233.51,"2,65249E+15",17 Years and 10 Months,No,"2,28018E+14",332.3337079767732,High_spent_Medium_value_payments,"4,8589E+15",01/12/2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12495,CUS_0xff3,17032.785,117639875,0,6,2,3,"Personal Loan, Mortgage Loan, and Auto Loan",13,7_,...,Good,1229.08,"2,69156E+15",17 Years and 3 Months,No,"3,32998E+16",81.19585741743609,Low_spent_Small_value_payments,"2,93144E+16",01/06/2024
12496,CUS_0xff4,25546.26,2415855,8,7,14,5_,"Not Specified, Student Loan, Student Loan, Cre...",15,13_,...,Standard,758.44,"3,93335E+15",18 Years and 9 Months,Yes,"1,01329E+15",189.81586133347676,Low_spent_Medium_value_payments,"2,30441E+16",01/12/2024
12497,CUS_0xff6,117639.92,"9,72733E+15",5,6,1,2,"Home Equity Loan, and Auto Loan",-3,7,...,Good,338.3,"3,28719E+15",24 Years and 11 Months,No,"1,26638E+16",534.0885271982645,Low_spent_Medium_value_payments,"5,92006E+15",01/10/2024
12498,CUS_0xffc,60877.17,52180975,6,8,27,8,"Credit-Builder Loan, Payday Loan, Not Specifie...",46,14,...,_,1300.13,"2,90265E+15",13 Years and 1 Months,Yes,"2,72809E+16",46.4256138380274,High_spent_Large_value_payments,"4,42575E+15",01/01/2024


In [5]:
def process_bronze_table(snapshot_date_str, bronze_clickstream_directory, bronze_attributes_directory, bronze_financials_directory, spark):
    snapshot_date = datetime.strptime(snapshot_date_str, "%Y-%m-%d")
    # snapshot_date_formatted = snapshot_date.strftime("%d/%m/%Y")

    # feature_clickstream
    clickstream_df = spark.read.csv("../data/feature_clickstream.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', clickstream_df.count())
    
    partition_name = "bronze_clickstream_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_clickstream_directory + partition_name
    clickstream_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    # features_attributes
    attributes_df = spark.read.csv("../data/features_attributes.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', attributes_df.count())
    
    partition_name = "bronze_attributes_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_attributes_directory + partition_name
    attributes_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    # features_financials
    financials_df = spark.read.csv("../data/features_financials.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', financials_df.count())
    
    partition_name = "bronze_financials_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_financials_directory + partition_name
    financials_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    return clickstream_df, attributes_df, financials_df